# Отчёт по ML-исследованию для защиты

Этот notebook — компактный презентационный отчёт по ML-блоку Game Intelligence Platform.

Перед запуском рекомендуется выполнить:

```bash
make er-merge-strategy-comparison
make er-graph-analysis
make er-embedding-research
make igdb-matching-analysis
make ml-research-defense
make bayesian-rating
make rag-explanations
make ml-defense-readiness
```

Для полного пересбора перед защитой используйте `make ml-defense-all`.


In [ ]:
import csv
import json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
REPORT_DIR = ROOT / 'data' / 'artifacts' / 'reports'
DEFENSE_DIR = REPORT_DIR / 'ml_research_defense'
ER_DIR = REPORT_DIR / 'entity_resolution'
GRAPH_DIR = REPORT_DIR / 'graph_analysis'
EMBEDDING_DIR = REPORT_DIR / 'embedding_research'
IGDB_DIR = REPORT_DIR / 'igdb_matching'
BAYESIAN_DIR = REPORT_DIR / 'bayesian_rating'
RAG_DIR = REPORT_DIR / 'rag_explanations'
READINESS_DIR = REPORT_DIR / 'ml_defense_readiness'

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def read_csv(path):
    with path.open(encoding='utf-8') as file:
        return list(csv.DictReader(file))

summary = read_json(DEFENSE_DIR / 'ml_research_defense_summary.json')
merge = read_json(ER_DIR / 'merge_strategies' / 'merge_strategy_comparison.json')
graph = read_json(GRAPH_DIR / 'graph_analysis_summary.json')
embedding = read_json(EMBEDDING_DIR / 'embedding_research_summary.json')
igdb = read_json(IGDB_DIR / 'igdb_matching_summary.json')
bayesian_summary = read_json(BAYESIAN_DIR / 'bayesian_rating_summary.json')
rag_summary = read_json(RAG_DIR / 'rag_explanation_summary.json')
readiness_summary = read_json(READINESS_DIR / 'ml_defense_readiness_summary.json')
summary.keys(), merge.keys(), graph.keys(), embedding.keys(), igdb.keys(), bayesian_summary.keys(), rag_summary.keys(), readiness_summary.keys()


## 1. Базовое состояние данных и ручной разметки

Этот раздел показывает, что в проекте достаточно данных и manual labels для сильного объяснимого baseline.

In [ ]:
summary['baseline_counts']


In [ ]:
summary['training_dataset']


## 2. Влияние качества данных

Data-quality artifacts объясняют, почему Entity Resolution требует ручной проверки и консервативных threshold-правил.

In [ ]:
summary['data_quality']


## 3. Метрики Entity Resolution

Ключевой тезис защиты: модель полезна как scoring/manual-review/controlled-hybrid слой, а не как слепой canonical merge.

In [ ]:
summary['existing_er_artifacts']['v3c_metrics']


In [ ]:
read_csv(ER_DIR / 'iterations' / 'manual_threshold_eval_v3c.csv')


## 4. Исследование ablation и calibration

Ablation показывает, какие группы признаков наиболее важны. Calibration проверяет, можно ли использовать probability scores для threshold policy.

In [ ]:
read_csv(DEFENSE_DIR / 'ablation_study.csv')


In [ ]:
summary['calibration']


### Графики для презентации

![Ablation F1](../data/artifacts/reports/ml_research_defense/charts/ablation_f1.svg)

![Calibration](../data/artifacts/reports/ml_research_defense/charts/calibration_bins.svg)

## 5. Сравнение стратегий объединения

Shadow comparison показывает, почему model-only auto-merge требует governance и не должен напрямую заменять trusted canonical layer.

In [ ]:
merge['strategies']


![F1 по merge strategies](../data/artifacts/reports/ml_research_defense/charts/merge_strategy_f1.svg)

## 6. Анализ ER-графа рисков

Graph risk analysis показывает транзитивный риск merge-ошибок: качество на уровне пар может выглядеть сильным, но merge policy всё равно может создавать same-source duplicate components или risky clusters.

In [ ]:
graph['strategies']


In [ ]:
read_csv(GRAPH_DIR / 'risky_components.csv')[:20]


In [ ]:
read_csv(GRAPH_DIR / 'high_probability_reviewed_negatives.csv')[:20]


![Same-source duplicate links](../data/artifacts/reports/graph_analysis/same_source_duplicate_links.svg)

## 7. Лёгкое исследование title embeddings

Локальный TF-IDF/SVD эксперимент сравнивает vector title similarity с fuzzy name similarity до добавления более тяжёлых neural embeddings.

In [ ]:
embedding


In [ ]:
read_csv(EMBEDDING_DIR / 'embedding_model_comparison.csv')


In [ ]:
read_csv(EMBEDDING_DIR / 'embedding_error_cases.csv')[:20]


![F1 embedding-моделей](../data/artifacts/reports/embedding_research/embedding_model_f1.svg)

## 8. Анализ IGDB-specific matching

Раздел оценивает IGDB как search-based источник ER candidates: качество retrieval rank, reviewed precision, high-risk negatives и enrichment coverage.

In [ ]:
igdb


In [ ]:
read_csv(IGDB_DIR / 'igdb_review_precision_by_rank.csv')


In [ ]:
read_csv(IGDB_DIR / 'igdb_enrichment_coverage.csv')


In [ ]:
read_csv(IGDB_DIR / 'igdb_high_risk_reviewed_negatives.csv')[:20]


![Распределение IGDB rank](../data/artifacts/reports/igdb_matching/igdb_rank_distribution.svg)

![Покрытие IGDB enrichment](../data/artifacts/reports/igdb_matching/igdb_enrichment_coverage.svg)

## 9. Active learning и примеры рекомендаций

Эти примеры полезны для живой защиты: какие пары модель предложила бы проверить следующими и как baseline recommendations объясняют общие признаки.

In [ ]:
read_csv(DEFENSE_DIR / 'active_learning_candidates.csv')[:20]


In [ ]:
read_csv(DEFENSE_DIR / 'recommendation_examples.csv')[:20]


## 10. Casebook для защиты и распределение recommendation score

Используйте эти строки как сценарий защиты: successful merges, rejected risky matches, active-learning pairs и recommendation examples.

In [ ]:
read_csv(DEFENSE_DIR / 'defense_demo_cases.csv')


In [ ]:
read_csv(DEFENSE_DIR / 'recommendation_score_distribution.csv')


![Распределение recommendation score](../data/artifacts/reports/ml_research_defense/charts/recommendation_score_distribution.svg)

## 11. Grounded RAG-like explanations

Раздел показывает русскоязычные explanations, сгенерированные только из computed facts. LLM/RAG-like layer не принимает решения по match или recommendation.

In [ ]:
rag_summary


In [ ]:
read_csv(RAG_DIR / 'match_explanation_examples.csv')[:10]


In [ ]:
read_csv(RAG_DIR / 'recommendation_explanation_examples.csv')[:10]


In [ ]:
read_csv(RAG_DIR / 'grounded_fact_cards.csv')[:10]


## 12. Анализ Bayesian rating

Bayesian rating — дополнительный статистический блок: он сравнивает naive source ratings с vote-adjusted ratings и показывает, почему игры с малым числом голосов нужно сдвигать к global mean.

In [ ]:
bayesian_summary


In [ ]:
read_csv(BAYESIAN_DIR / 'canonical_bayesian_ratings.csv')[:20]


In [ ]:
read_csv(BAYESIAN_DIR / 'low_vote_shrinkage_examples.csv')[:20]


![Top Bayesian ratings](../data/artifacts/reports/bayesian_rating/top_bayesian_ratings.svg)

## 13. Финальный readiness gate перед защитой

Этот раздел проверяет, что все обязательные research artifacts и ключевые метрики доступны перед репетицией защиты.

In [ ]:
readiness_summary


In [ ]:
read_csv(READINESS_DIR / 'ml_defense_metric_snapshot.csv')


In [ ]:
read_csv(READINESS_DIR / 'ml_defense_demo_sequence.csv')


In [ ]:
read_csv(READINESS_DIR / 'ml_defense_artifact_checklist.csv')


## 14. Выводы для защиты

- Entity Resolution — самый сильный текущий ML-исследовательский блок.
- Manual labels критичны: они дают и positive, и negative examples.
- `model_auto_090` полезен для high-confidence candidates, но не как blind canonical merge.
- `model_auto_070_research` полезен для risk analysis, потому что выявляет false positives и cluster problems.
- Graph risk analysis показывает транзитивный ER-риск, который не виден только по pair-level metrics.
- Лёгкие title embeddings улучшают fuzzy-only baseline без скачивания внешних моделей.
- IGDB rank analysis показывает, почему search candidates требуют retrieval governance и manual-review sampling.
- Recommendations — объяснимый content-based baseline, его стоит показывать как вторичный ML-блок.
- Grounded RAG-like explanations только визуализируют computed facts; они не принимают решения по merge или recommendations.
- Bayesian rating демонстрирует отдельный интерпретируемый статистический блок для robust ranking при sparse vote counts.